# 🌟 06 — Cosmos 3: The Reasoner (Understanding, Leveled Up)

> **Goal:** use NVIDIA's newest model, **Cosmos 3**, to understand video — captions,
> timelines, physical-plausibility checks, and more.

---

## What's different about Cosmos 3?

Cosmos-Reason2 (notebooks 01–05) runs the model **inside Python**. Cosmos 3 is bigger and
smarter, so it runs as a **separate server** (vLLM) that your agent talks to over the
network. Think of it like calling a powerful service running on the same machine.

```mermaid
flowchart LR
    AG["🤖 Agent +<br/>Cosmos3ReasonerModel"]:::agent
    NET{{"🌐 http://localhost:8000"}}:::data
    SRV["🟢 vLLM server<br/>Cosmos3-Nano (16B)"]:::reason
    OUT([💬 rich understanding]):::out
    AG -->|"&lt;video&gt; question"| NET --> SRV --> OUT
    classDef user fill:#ECEFF1,stroke:#607D8B,stroke-width:2px,color:#263238
    classDef agent fill:#E3F2FD,stroke:#1976D2,stroke-width:2px,color:#0D47A1
    classDef reason fill:#E8F5E9,stroke:#388E3C,stroke-width:2px,color:#1B5E20
    classDef gen fill:#F3E5F5,stroke:#8E24AA,stroke-width:2px,color:#4A148C
    classDef tool fill:#FFF3E0,stroke:#F57C00,stroke-width:2px,color:#E65100
    classDef data fill:#FFFDE7,stroke:#FBC02D,stroke-width:2px,color:#F57F17
    classDef out fill:#E0F2F1,stroke:#00897B,stroke-width:2px,color:#004D40
    classDef think fill:#FCE4EC,stroke:#D81B60,stroke-width:2px,color:#880E4F
```

!!! info "One-time setup"
    ```bash
    just c3-doctor          # check GPU / CUDA / disk
    just c3-setup-reason    # build the reasoner environment (vLLM)
    just c3-serve-reason    # serve Cosmos3-Nano on :8000
    ```
    The preflight cell below tells you if the server is already running.

In [ ]:
# 🔧 Preflight for Cosmos 3 — checks GPU *and* whether a reasoner server is up.
import os, sys, time
os.environ.setdefault("QWEN_VL_VIDEO_READER", "torchcodec")
sys.path.insert(0, os.path.abspath(".."))

# 🔒 Cosmos tools confine file access to a workspace allow-list for safety.
# The notebooks live in notebooks/ but our sample media sits one level up, so we
# explicitly widen the workspace to the project root + /tmp. (This is how you grant
# the tools access to a folder — never wider than you need.)
os.environ["COSMOS_WORKSPACE"] = os.pathsep.join([os.path.abspath(".."), "/tmp"])

PROJECT_ROOT = os.path.abspath("..")
SAMPLE_VIDEO = os.path.join(PROJECT_ROOT, "sample.mp4")
REASONER_URL = os.environ.get("C3_BASE_URL", "http://localhost:8000/v1")

def gpu_ready():
    try:
        import torch; return torch.cuda.is_available()
    except Exception:
        return False

def server_up(url):
    import urllib.request
    try:
        with urllib.request.urlopen(url.rstrip("/") + "/models", timeout=2) as r:
            return r.status == 200
    except Exception:
        return False

GPU = gpu_ready()
SERVER = server_up(REASONER_URL)
print("🎮 GPU available  :", GPU)
print("🟢 Reasoner server:", SERVER, f"({REASONER_URL})")
if not SERVER:
    print("   ↳ Start one with:  just c3-setup-reason && just c3-serve-reason")
print("📹 sample.mp4     :", os.path.exists(SAMPLE_VIDEO))

## Step 1 — Connect to the reasoner
No model download here — we just point at the running server. If it's not up, we skip.

In [ ]:
agent = None
if SERVER:
    from strands import Agent
    from strands_cosmos import Cosmos3ReasonerModel
    model = Cosmos3ReasonerModel(base_url=REASONER_URL, max_tokens=512)
    agent = Agent(model=model)
    print("✅ Connected to the Cosmos 3 reasoner")
else:
    print("⏭  No server — start it with: just c3-serve-reason")

## Step 2 — Detailed caption
Same `<video>` tag you already know — now powered by a much stronger model.

In [ ]:
if agent is not None:
    t0 = time.time()
    agent(f"Caption in detail: <video>{SAMPLE_VIDEO}</video>")
    print(f"\n⏱  {time.time()-t0:.1f}s")
else:
    print("⏭  Skipped.")

## Step 3 — The specialist tools
Cosmos 3 ships ready-made tools for common video questions. Each is one function call.
They also need the server, so we guard them together.

In [ ]:
from strands_cosmos import cosmos3_caption, cosmos3_temporal, cosmos3_plausibility

if SERVER:
    print("— Timeline of events —")
    print(cosmos3_temporal(video=SAMPLE_VIDEO)["content"][0]["text"][:400])
    print("\n— Is it physically plausible? —")
    print(cosmos3_plausibility(video=SAMPLE_VIDEO)["content"][0]["text"][:400])
else:
    print("⏭  These call the server too — skipped.")

## Step 4 — Turn on deep reasoning
Like before, you can ask for step-by-step thinking for the hard questions.

In [ ]:
if agent is not None:
    model.update_config(reasoning=True)
    agent(f"What is the most likely next action in this scene? <video>{SAMPLE_VIDEO}</video>")
else:
    print("⏭  Skipped.")

# 🎓 What you learned

| Concept | Takeaway |
|---------|----------|
| `Cosmos3ReasonerModel` | Talks to a **vLLM server** (not in-process) |
| `base_url` | Where the server lives (`localhost:8000`) |
| `cosmos3_caption / _temporal / _plausibility` | Ready-made specialist tools |
| Same `<video>` tag | The interface never changes |

**Next:** [07 — Cosmos 3: Generate](07_cosmos3_generate.ipynb) — the other half of the
brain: **creating** video and sound. 🟣 →